<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">1. DataFrame API Basics</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 1.2 Optional Demo - Basic ETL with the DataFrame API
### Duration - 10 Minutes

This optional demonstration walks through common ETL operations using the Bakehouse sales transactions sample dataset. We cover data loading, validation, transformation, and enrichment using the DataFrame API. If you are comfortable with this material from the *Introduction to Apache Spark* course, you can skim it as a refresher.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Implement common ETL operations using Spark DataFrames
- Validate data quality with both Spark SQL and the DataFrame API
- Create derived features through transformations
- Use different column reference methods
- Work with user defined functions (UDFs)

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run, sets your default catalog, and sets the default schema to `spark_dev`.

In [0]:
%run ../Includes/Classroom-Setup-1

### Transaction Data Processing Requirements

We are going to act on a typical ETL requirement: land a validated, enriched version of the raw transaction data as a new table.

**Source data** - the `samples.bakehouse.sales_transactions` table (point-of-sale transactions from the Bakehouse sample dataset)

**Target** - a table named `transactions_enriched` with this schema:

| Column Name | Data Type | Description |
|-------------|-----------|-------------|
| __`transactionID`__ | `BIGINT` | Transaction identifier |
| __`dateTime`__ | `TIMESTAMP` | When the transaction occurred |
| __`product`__ | `STRING` | Product sold |
| __`quantity`__ | `BIGINT` | Units sold |
| __`totalPrice`__ | `BIGINT` | Total sale amount |
| __`paymentMethod`__ | `STRING` | Payment method, normalized to uppercase |
| __`cardNumberMasked`__ | `STRING` | Card number with all but the last 4 digits masked |
| __`orderSize`__ | `STRING` | One of `Bulk`, `Large`, `Medium`, `Small`, based on `quantity` |

## B. Data Loading and Inspection

First, let's load and inspect the transaction data.

In [0]:
%python
# Read the Transactions Table
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

In [0]:
%python
# Print the Schema
transactions_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The schema was read from the table's Unity Catalog metadata - no inference required. Note that <code>cardNumber</code> arrived as a <code>BIGINT</code>: storing a card number as a number is a data quality smell, and we will convert and mask it during enrichment.</p>
</div>

In [0]:
%python
# Visually Inspect a Subset of the Data
display(transactions_df.limit(10))

In [0]:
%python
# Select Only the Columns We Need
# Remember: filter early, filter often
transactions_required_cols_df = transactions_df.select(
    "transactionID",
    "dateTime",
    "product",
    "quantity",
    "unitPrice",
    "totalPrice",
    "paymentMethod",
    "cardNumber")

# Alternatively we could have used the drop() method to remove the columns we didn't want

In [0]:
%python
# Count the Source Records
initial_count = transactions_required_cols_df.count()

print(f"Source data has {initial_count} records")

## C. Data Validation

Before enriching the data we check it for nulls and internal consistency - here, that the line total equals `quantity * unitPrice`.

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#9888;&#65039;</span>
<div>
<strong style="color: #e65100; font-size: 1.1em">ANSI Mode and Casting</strong>
<p style="margin: 8px 0 0 0; color: #333">Spark 4 runs in ANSI SQL mode by default: <code>CAST('NA' AS INT)</code> raises an error instead of silently returning <code>NULL</code>. The Bakehouse sample data is clean, but when you process data of unknown quality, use <code>TRY_CAST</code>, which returns <code>NULL</code> for invalid values, letting you find and handle them deliberately.</p>
</div>
</div>
</div>

In [0]:
%python
# Register a Temp View for SQL Analysis
transactions_required_cols_df.createOrReplaceTempView("transactions_temp")

In [0]:
%python
# Count Null Values with Spark SQL
# Use the COUNT_IF SQL function to count null values per column
null_counts_sql = spark.sql('''
SELECT
    COUNT_IF(transactionID IS NULL) AS Null_TransactionID_Count,
    COUNT_IF(dateTime IS NULL) AS Null_DateTime_Count,
    COUNT_IF(product IS NULL) AS Null_Product_Count,
    COUNT_IF(quantity IS NULL) AS Null_Quantity_Count,
    COUNT_IF(unitPrice IS NULL) AS Null_UnitPrice_Count,
    COUNT_IF(totalPrice IS NULL) AS Null_TotalPrice_Count,
    COUNT_IF(paymentMethod IS NULL) AS Null_PaymentMethod_Count,
    COUNT_IF(cardNumber IS NULL) AS Null_CardNumber_Count
FROM transactions_temp
''')

display(null_counts_sql)

### Comparing Spark SQL to DataFrame API Operations

Spark SQL queries and their equivalent operations in the DataFrame API are evaluated to the same physical plans. Let's prove this.

In [0]:
%python
# Count Null Values with the DataFrame API
# This is the equivalent of the preceding Spark SQL query using the DataFrame API
from pyspark.sql.functions import col, sum, when

# Make sure to work with the same temporary view that the SQL is using
transactions_temp_df = spark.table("transactions_temp")

null_counts_df = transactions_temp_df.select(
    sum(when(col("transactionID").isNull(), 1).otherwise(0)).alias("Null_TransactionID_Count"),
    sum(when(col("dateTime").isNull(), 1).otherwise(0)).alias("Null_DateTime_Count"),
    sum(when(col("product").isNull(), 1).otherwise(0)).alias("Null_Product_Count"),
    sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("Null_Quantity_Count"),
    sum(when(col("unitPrice").isNull(), 1).otherwise(0)).alias("Null_UnitPrice_Count"),
    sum(when(col("totalPrice").isNull(), 1).otherwise(0)).alias("Null_TotalPrice_Count"),
    sum(when(col("paymentMethod").isNull(), 1).otherwise(0)).alias("Null_PaymentMethod_Count"),
    sum(when(col("cardNumber").isNull(), 1).otherwise(0)).alias("Null_CardNumber_Count")
)

display(null_counts_df)

In [0]:
%python
# Explain the SQL Version
null_counts_sql.explain()

In [0]:
%python
# Explain the DataFrame Version
null_counts_df.explain()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Compare the two physical plans printed above - they are the same. Spark SQL and the DataFrame API are two front ends to the same Catalyst optimizer and execution engine, so you can choose whichever is clearer for the task at hand.</p>
</div>

In [0]:
%python
# Check Line Total Consistency
from pyspark.sql.functions import col

# Keep only rows where the line total is consistent with quantity * unitPrice
valid_transactions_df = transactions_required_cols_df.filter(
    col("quantity") > 0
).filter(
    col("totalPrice") == col("quantity") * col("unitPrice")
)

print(f"{valid_transactions_df.count()} of {initial_count} records passed validation")

## D. Data Enrichment

Now let's create the derived columns from the requirements: a normalized payment method, a masked card number, and an order size category.

In [0]:
%python
# Normalize the Payment Method
from pyspark.sql.functions import upper, col

normalized_df = valid_transactions_df.withColumn(
    "paymentMethod", upper(col("paymentMethod"))
)

display(normalized_df.select("transactionID", "paymentMethod").limit(10))

In [0]:
%python
# Mask the Card Number
from pyspark.sql.functions import col, concat, lit, right

# Cast the card number to a string, then keep only the last 4 digits
masked_df = normalized_df.withColumn(
    "cardNumberMasked",
    concat(lit("****-"), right(col("cardNumber").cast("string"), lit(4)))
).drop("cardNumber")

display(masked_df.select("transactionID", "cardNumberMasked").limit(10))

In [0]:
%python
# Categorize the Order Size
from pyspark.sql.functions import when, col

# Categorize the quantity column into: Bulk, Large, Medium, Small
enriched_df = masked_df.withColumn(
    "orderSize",
    when(col("quantity") >= 24, "Bulk")
    .when(col("quantity") >= 12, "Large")
    .when(col("quantity") >= 6, "Medium")
    .otherwise("Small")
).drop("unitPrice")

display(enriched_df.limit(10))

### Visualize the Order Sizes

PySpark in Spark 4 includes native plotting: call `df.plot` directly on a DataFrame (backed by plotly) without converting to pandas first.

In [0]:
%python
# Plot Order Sizes with Native Plotting
# Aggregate, then plot directly from the PySpark DataFrame (new in Spark 4)
order_size_counts_df = enriched_df.groupBy("orderSize").count()

order_size_counts_df.plot.bar(x="orderSize", y="count")

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Other Visualization Options</strong>
<p style="margin: 8px 0 0 0; color: #333">You can also use the <strong>+</strong> -> <strong>Visualization</strong> option in any <code>display()</code> result, or ask the Databricks Assistant to generate chart code for you.</p>
</div>
</div>
</div>

## E. Working with Different Column References

Let's analyze the enriched data while exercising the column referencing approaches from the lecture.

In [0]:
%python
# Reference Columns Directly
# Direct string references
display(enriched_df.select("product", "orderSize").limit(100))

In [0]:
%python
# Reference Columns with Column Objects
from pyspark.sql.functions import col

# Column objects support methods like alias()
display(
    enriched_df
    .select(col("product").alias("product_name"), col("totalPrice"))
    .limit(100)
)

In [0]:
%python
# Reference Columns with String Expressions
# selectExpr accepts SQL expressions
display(
    enriched_df
    .selectExpr("product", "totalPrice", "totalPrice >= 100 AS isLargeSale")
    .limit(100)
)

## F. Working with UDFs

Let's use a vectorized pandas UDF to calculate the z-score (standard deviations from the mean) of the total price for each transaction.

In [0]:
%python
# Apply a Vectorized Pandas UDF
from pyspark.sql.functions import pandas_udf

# Pandas UDF (vectorized) - operates on batches of rows as pandas Series
@pandas_udf("double")
def normalized_price(price_series):
    return (price_series - price_series.mean()) / price_series.std()

udf_example_df = enriched_df.withColumn(
    "price_normalized", normalized_price("totalPrice")
)

display(udf_example_df.limit(100))

# Note: in practice, prefer built-in functions over UDFs when possible

## G. Putting It All Together

Let's combine everything into a single chained operation that reads from the source, validates, enriches, and saves to a new target table (overwriting any existing data).

In [0]:
-- Drop the Target Table if it Exists
DROP TABLE IF EXISTS transactions_enriched;

In [0]:
%python
# Run the Full ETL Pipeline
from pyspark.sql.functions import col, concat, lit, right, upper, when

(spark.read.table("samples.bakehouse.sales_transactions")
    # Project the columns we need
    .select(
        "transactionID",
        "dateTime",
        "product",
        "quantity",
        "unitPrice",
        "totalPrice",
        "paymentMethod",
        "cardNumber")
    # Keep only internally consistent rows
    .filter(col("quantity") > 0)
    .filter(col("totalPrice") == col("quantity") * col("unitPrice"))
    # Normalize the payment method
    .withColumn("paymentMethod", upper(col("paymentMethod")))
    # Mask the card number
    .withColumn(
        "cardNumberMasked",
        concat(lit("****-"), right(col("cardNumber").cast("string"), lit(4)))
    )
    .drop("cardNumber")
    # Categorize the order size
    .withColumn(
        "orderSize",
        when(col("quantity") >= 24, "Bulk")
        .when(col("quantity") >= 12, "Large")
        .when(col("quantity") >= 6, "Medium")
        .otherwise("Small")
    )
    .drop("unitPrice")
    # Write to the target table
    .write
    .mode("overwrite")
    .saveAsTable("transactions_enriched"))

In [0]:
-- Query the Target Table
SELECT * FROM transactions_enriched LIMIT 10;

## Key Takeaways

**Data validation best practices:**

- Project only the columns you need, as early as possible
- Check for nulls and internal consistency before enriching
- Under Spark 4 ANSI mode, use `try_cast()` when converting data of unknown quality

**Data enrichment:**

- Create meaningful derived columns driven by business requirements
- Use built-in functions where possible, and user defined functions where necessary

**Pipeline composition:**

- Because transformations are lazy and return new DataFrames, an entire pipeline can be expressed as one readable chain ending in a single write action

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>